# NB15 — Estructura espectral, errores y sensibilidad de la incertidumbre

**Preguntas:** ¿qué cambia entre el prefijo y las observaciones posteriores? ¿dónde se concentran los errores? ¿cuánto depende el resumen de un experimento? ¿cambian los intervalos al remuestrear firmas completas en vez de filas?

CPU. No entrena clasificadores ni cambia NB14. Lee las predicciones guardadas. Genera tablas, PCA, cargas espectrales, UMAP, gráficos de errores, sensibilidades y un ZIP al final.

Población primaria: las mismas 1.795 asociaciones firma–etapa de NB14 para N=25. Representaciones: RAW, SNV por observación, PREFIX y FULL_TARGET. SNV y FULL_TARGET son diagnósticos visuales; no se les atribuyen las predicciones del brazo PREFIX. FULL_TARGET utiliza datos sin etiquetas del lote completo y es transductivo. PCA/UMAP se ajustan solo con los experimentos de desarrollo; el test se proyecta sin reajuste.

Los índices no son fechas validadas. Las cargas PCA no son importancia predictiva. Las distancias de UMAP entre ajustes separados no son comparables. Los intervalos de filas o firmas son sensibilidades condicionales; ninguna de las dos garantiza corrección frente a dependencia temporal o por fruto. Solo tenemos cuatro experimentos externos, con conjuntos de entrenamiento solapados. No se añaden pruebas poblacionales de significación.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import sys, subprocess, importlib.util
if importlib.util.find_spec('umap') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','umap-learn==0.5.12'])
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, zipfile, io, shutil, platform
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from IPython.display import display
import umap
PROJECT=Path('/content/drive/MyDrive/Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1')
MASTER=PROJECT/'02_EXTRACTED_DATA/06_master_original_multisensor.csv'
RESULTS=PROJECT/'06_RESULTS'
# Set an exact ZIP path here to override automatic selection.
INPUT_ZIP=None
N=25; B=1000; SEED=20261025
RUN=RESULTS/('NB15_SPECTRAL_ERROR_AUDIT_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
TABLE=RUN/'tables'; FIG=RUN/'figures'; META=RUN/'metadata'
for p in [TABLE,FIG,META]:p.mkdir(parents=True,exist_ok=False)
if INPUT_ZIP is None:
    candidates=sorted(RESULTS.glob('RESULTS_NB14_PROSPECTIVE_DOMAIN_Z_*.zip'),key=lambda p:p.stat().st_mtime,reverse=True)
    if not candidates:raise FileNotFoundError('NB14 ZIP not found in 06_RESULTS. Upload the completed NB14 ZIP there.')
    INPUT_ZIP=candidates[0]
INPUT_ZIP=Path(INPUT_ZIP)
FEATURES=['SpA410','SpB435','SpC460','SpD485','SpE510','SpF535','SpG560','SpH585','SpR610','SpI645','SpS680','SpJ705','SpT730','SpU760','SpV810','SpW860','SpK900','SpL940']
WAVELENGTHS=[410,435,460,485,510,535,560,585,610,645,680,705,730,760,810,860,900,940]
TESTS=[2,3,4,5]; ARMS=['RAW','SNV','PREFIX','FULL_TARGET']
plt.rcParams.update({'font.size':11,'pdf.fonttype':42,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False})
def savefig(fig,name):
    fig.tight_layout()
    for ext in ['png','pdf','svg']:fig.savefig(FIG/f'{name}.{ext}',dpi=400,bbox_inches='tight')
    plt.show();plt.close(fig)
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
print('Input:',INPUT_ZIP,'\nOutput:',RUN,'\nCPU only; no classifier training.')


In [ ]:
# 1. Validate source archive and hashes before reading analysis files.
assert sha(MASTER)=='0a06fb3773f85dacee42e18eecf17e5e2a8e7f5e9508cd3d60bd4b66757e8ea8','Master differs from frozen source'
z=zipfile.ZipFile(INPUT_ZIP)
assert z.testzip() is None
root_candidates=[p for p in z.namelist() if p.endswith('/metadata/run_manifest.json')]
assert len(root_candidates)==1
BASE=root_candidates[0].removesuffix('metadata/run_manifest.json')
def zread(p):return z.read(BASE+p)
def zcsv(p):return pd.read_csv(io.BytesIO(zread(p)),compression='gzip' if p.endswith('.gz') else None)
hashes=zcsv('metadata/package_hashes.csv')
for r in hashes.to_dict('records'):
    assert hashlib.sha256(zread(r['relative_path'])).hexdigest()==r['sha256'],r['relative_path']
manifest=json.loads(zread('metadata/run_manifest.json'))
assert manifest['primary_N']==25
p=zcsv('tables/selected_predictions_paired_diagnostics.csv.gz')
p=p[p.N_prefix.eq(N)].copy()
assert set(p.arm)=={'RAW_FUTURE','DOMAIN_Z_PREFIX'}
assert not p[['outer_test','arm','master_id']].duplicated().any()
wide=p.pivot(index=['outer_test','master_id','stage_true'],columns='arm',values='stage_pred').reset_index()
assert wide[['RAW_FUTURE','DOMAIN_Z_PREFIX']].notna().all().all()
wide.columns.name=None
print('Integrity verified:',len(hashes),'files;',len(wide),'paired primary observations')


In [ ]:
# 2. Reconstruct exact NB14 evaluation population and prefix statistics.
df=pd.read_csv(MASTER)
df['acq_order_proxy']=(df.spect_source_index.combine_first(df.ds34_source_index).combine_first(df.tphis_source_index).combine_first(df.master_id))
df['signature']=df.groupby(FEATURES,sort=False,dropna=False).ngroup()
shared=df.groupby('signature').test_id.nunique();shared=set(shared[shared>1].index)
assert len(shared)==2
clean=df[~df.signature.isin(shared)].sort_values(['test_id','acq_order_proxy','master_id']).copy()
dedup=clean.drop_duplicates(['test_id']+FEATURES+['stage'],keep='first').copy()
assert len(dedup)==1895
prefix={};full={};future_parts=[];prefix_audit=[]
def stats(X):
    X=np.asarray(X,float);sd=X.std(axis=0);sd=np.where(sd<1e-12,1.,sd)
    return X.mean(axis=0),sd
for t in TESTS:
    g=clean[clean.test_id.eq(t)]
    u=g.drop_duplicates(FEATURES,keep='first')
    q=u.head(N);prefix[t]=stats(q[FEATURES]);full[t]=stats(u[FEATURES])
    f=dedup[dedup.test_id.eq(t)&~dedup.signature.isin(q.signature)&dedup.acq_order_proxy.gt(q.acq_order_proxy.max())].copy()
    future_parts.append(f)
    prefix_audit.append({'test_id':t,'prefix_n':len(q),'evaluated_n':len(f),'prefix_max_index':q.acq_order_proxy.max(),**{f'prefix_stage_{k}':int(q.stage.eq(k).sum()) for k in range(1,6)}})
future=pd.concat(future_parts,ignore_index=True)
assert set(future.master_id)==set(wide.master_id),'Reconstructed evaluation IDs differ from NB14'
assert not future.master_id.duplicated().any()
a=wide.merge(future[['master_id','test_id','stage','signature']],on='master_id',validate='one_to_one')
assert np.array_equal(a.stage_true,a.stage) and np.array_equal(a.outer_test,a.test_id)
pd.DataFrame(prefix_audit).to_csv(TABLE/'prefix_support_audit.csv',index=False)
wide=a
for arm,col in [('raw','RAW_FUTURE'),('prefix','DOMAIN_Z_PREFIX')]:wide[f'{arm}_ae']=(wide[col]-wide.stage_true).abs()
wide['delta_ae']=wide.prefix_ae-wide.raw_ae
wide.to_csv(TABLE/'paired_predictions_with_signature.csv.gz',index=False,compression='gzip')
def transformed(g,arm):
    X=g[FEATURES].to_numpy(float)
    if arm=='RAW':return X
    if arm=='SNV':
        sd=X.std(axis=1,keepdims=True);return (X-X.mean(axis=1,keepdims=True))/np.where(sd<1e-12,1.,sd)
    X=X.copy()
    for t in TESTS:
        mask=g.test_id.eq(t).to_numpy()
        if mask.any():
            mu,sd=(prefix if arm=='PREFIX' else full)[t];X[mask]=(X[mask]-mu)/sd
    return X
print('Exact NB14 cohort reproduced:',len(future))


In [ ]:
# 3. Per-experiment metrics and leave-one-experiment-out summary sensitivity.
metrics=[]
for t,g in wide.groupby('outer_test'):
    labels=sorted(g.stage_true.unique())
    metrics.append({'outer_test':t,'n':len(g),'n_signatures':g.signature.nunique(),
      'raw_f1':f1_score(g.stage_true,g.RAW_FUTURE,labels=labels,average='macro',zero_division=0),
      'prefix_f1':f1_score(g.stage_true,g.DOMAIN_Z_PREFIX,labels=labels,average='macro',zero_division=0),
      'delta_mae':g.delta_ae.mean(),'delta_severe':g.prefix_ae.ge(2).mean()-g.raw_ae.ge(2).mean()})
metrics=pd.DataFrame(metrics);metrics['delta_f1']=metrics.prefix_f1-metrics.raw_f1
metrics.to_csv(TABLE/'metrics_by_experiment.csv',index=False)
loo=[]
for omitted in TESTS:
    g=metrics[metrics.outer_test.ne(omitted)]
    loo.append({'omitted_test':omitted,'retained_experiments':len(g),'mean_delta_f1':g.delta_f1.mean(),'mean_delta_mae':g.delta_mae.mean(),'mean_delta_severe':g.delta_severe.mean()})
loo=pd.DataFrame(loo);loo.to_csv(TABLE/'leave_one_experiment_out_summary.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,col,title in zip(axes,['mean_delta_f1','mean_delta_mae'],['Macro-F1 change','Ordinal MAE change']):
    ax.plot(loo.omitted_test,loo[col],marker='o');ax.axhline(0,color='black',ls='--',lw=1)
    ax.set(xlabel='Omitted experiment',ylabel='PREFIX minus RAW',xticks=TESTS,title=title)
savefig(fig,'F15_01_Leave_One_Experiment_Out')
display(metrics);display(loo)


In [ ]:
# 4. Conditional resampling sensitivity: signatures vs rows, on fixed predictions.
# Signatures are sensor-state groups, not validated independent biological units.
def endpoints(g,idx):
    y=g.stage_true.to_numpy()[idx];r=g.RAW_FUTURE.to_numpy()[idx];p=g.DOMAIN_Z_PREFIX.to_numpy()[idx]
    labels=sorted(g.stage_true.unique())
    return np.array([f1_score(y,p,labels=labels,average='macro',zero_division=0)-f1_score(y,r,labels=labels,average='macro',zero_division=0),np.mean(np.abs(p-y)-np.abs(r-y)),np.mean(np.abs(p-y)>=2)-np.mean(np.abs(r-y)>=2)])
boot_records=[]
for t,g in wide.groupby('outer_test'):
    g=g.reset_index(drop=True);rng=np.random.default_rng(SEED+int(t))
    groups=[np.flatnonzero(g.signature.to_numpy()==s) for s in g.signature.unique()]
    strata=[np.flatnonzero(g.stage_true.to_numpy()==k) for k in sorted(g.stage_true.unique())]
    for method in ['stage_stratified_rows','signature_clusters']:
        draws=[]
        for b in range(B):
            if method=='stage_stratified_rows':idx=np.concatenate([rng.choice(s,len(s),replace=True) for s in strata])
            else:idx=np.concatenate([groups[i] for i in rng.integers(0,len(groups),len(groups))])
            draws.append(endpoints(g,idx))
        draws=np.array(draws);point=endpoints(g,np.arange(len(g)))
        for j,m in enumerate(['delta_macro_f1','delta_ordinal_mae','delta_severe_rate']):
            boot_records.append({'outer_test':t,'method':method,'metric':m,'point':point[j],'q025':np.quantile(draws[:,j],.025),'q975':np.quantile(draws[:,j],.975),'B':B,'n_signatures':len(groups),'scope':'Conditional sensitivity only; temporal/fruit dependence unresolved; fixed model predictions'})
boot=pd.DataFrame(boot_records);boot.to_csv(TABLE/'conditional_resampling_sensitivity.csv',index=False)
fig,ax=plt.subplots(figsize=(8,5))
for offset,(method,color) in enumerate([('stage_stratified_rows','#999999'),('signature_clusters','#0072B2')]):
    g=boot[(boot.method==method)&boot.metric.eq('delta_macro_f1')].sort_values('outer_test')
    yy=np.arange(4)+offset*.18
    # Draw interval endpoints directly; percentile intervals need not bracket point estimates.
    ax.hlines(yy,g.q025,g.q975,color=color,label=method)
    ax.scatter(g.point,yy,color=color)
ax.axvline(0,color='black',ls='--');ax.set_yticks(np.arange(4)+.09,[f'Test {t}' for t in TESTS]);ax.set_xlabel('Macro-F1: PREFIX minus RAW');ax.set_title('Conditional resampling sensitivity — not population inference');ax.legend(frameon=False,fontsize=9)
savefig(fig,'F15_02_Conditional_Intervals');display(boot)


In [ ]:
# 5. PCA and UMAP fitted on development only; external evaluation rows transformed.
# No settings selected by visual separation or prediction performance.
loads=[];variance=[];drift=[];coords=[]
for t in TESTS:
    dev=future[future.test_id.ne(t)].reset_index(drop=True);test=future[future.test_id.eq(t)].reset_index(drop=True)
    fig_p,axp=plt.subplots(2,4,figsize=(17,8));fig_l,axl=plt.subplots(1,4,figsize=(17,4))
    fig_u,axu=plt.subplots(2,4,figsize=(17,8));fig_e,axe=plt.subplots(1,2,figsize=(10,4))
    for j,arm in enumerate(ARMS):
        D=transformed(dev,arm);T=transformed(test,arm)
        scaler=StandardScaler().fit(D);D=scaler.transform(D);T=scaler.transform(T)
        pca=PCA(n_components=3,svd_solver='full').fit(D)
        # Anchor component signs for display only, not substantive cross-fit equivalence.
        for k in range(3):
            anchor=np.argmax(np.abs(pca.components_[k]))
            if pca.components_[k,anchor]<0:pca.components_[k]*=-1
        ds=pca.transform(D);ts=pca.transform(T)
        for k in range(3):
            variance.append({'external_test':t,'arm':arm,'PC':k+1,'explained_variance_ratio':pca.explained_variance_ratio_[k]})
            for band,v in zip(WAVELENGTHS,pca.components_[k]):loads.append({'external_test':t,'arm':arm,'PC':k+1,'wavelength_nm':band,'coefficient_standardized_space':v})
            axl[j].plot(WAVELENGTHS,pca.components_[k],marker='.',label=f'PC{k+1} {100*pca.explained_variance_ratio_[k]:.1f}%')
        axl[j].set(title=arm,xlabel='Wavelength (nm)',ylabel='PCA coefficient');axl[j].axhline(0,color='grey',lw=.5);axl[j].legend(fontsize=8,frameon=False)
        reducer=umap.UMAP(n_neighbors=30,min_dist=.15,n_components=2,metric='euclidean',random_state=SEED,transform_seed=SEED,n_jobs=1)
        du=reducer.fit_transform(D);tu=reducer.transform(T)
        allg=pd.concat([dev,test],ignore_index=True);domain=np.r_[dev.test_id.to_numpy(),test.test_id.to_numpy()];stage=allg.stage.to_numpy()
        for axes,Z,method in [(axp,np.vstack([ds[:,:2],ts[:,:2]]),'PCA'),(axu,np.vstack([du,tu]),'UMAP')]:
            for row,(color,title) in enumerate([(domain,'Experiment'),(stage,'Ripeness stage')]):
                axes[row,j].scatter(Z[:len(dev),0],Z[:len(dev),1],c=color[:len(dev)],cmap='tab10' if row==0 else 'viridis',vmin=1,vmax=5,s=8,alpha=.25)
                sc=axes[row,j].scatter(Z[len(dev):,0],Z[len(dev):,1],c=color[len(dev):],cmap='tab10' if row==0 else 'viridis',vmin=1,vmax=5,s=22,marker='^',edgecolors='black',linewidths=.25)
                axes[row,j].set(title=f'{arm} | {title}',xlabel=f'{method} 1',ylabel=f'{method} 2')
                fig=(fig_p if method=='PCA' else fig_u);fig.colorbar(sc,ax=axes[row,j],ticks=TESTS if row==0 else range(1,6),fraction=.045)
        for phase,gg,zz,uu in [('development',dev,ds,du),('external',test,ts,tu)]:
            for i,r in gg.iterrows():coords.append({'external_test':t,'arm':arm,'role':phase,'master_id':int(r.master_id),'stage':int(r.stage),'test_id':int(r.test_id),'PC1':zz[i,0],'PC2':zz[i,1],'UMAP1':uu[i,0],'UMAP2':uu[i,1]})
        # Mean displacement in development-standardized feature units; descriptive, not causal.
        drift.append({'external_test':t,'arm':arm,'mean_standardized_displacement':float(np.linalg.norm(T.mean(axis=0)-D.mean(axis=0))), 'scope':'future-row development-standardized feature space'})
        if arm=='PREFIX':
            err=test[['master_id']].merge(wide[['master_id','prefix_ae']],on='master_id',validate='one_to_one')
            category=np.minimum(err.prefix_ae.to_numpy(),2)
            for ax,Z,label in [(axe[0],ts[:,:2],'PCA'),(axe[1],tu,'UMAP')]:
                for k,color,lbl in [(0,'#009E73','Correct'),(1,'#E69F00','1-stage error'),(2,'#D55E00','Error >=2 stages')]:
                    mask=category==k;ax.scatter(Z[mask,0],Z[mask,1],s=22,color=color,label=lbl,alpha=.65)
                ax.set(title=f'PREFIX | {label} | Test {t}',xlabel=f'{label} 1',ylabel=f'{label} 2');ax.legend(frameon=False,fontsize=8)
    fig_p.suptitle(f'PCA | held-out Test {t} triangles; development circles');savefig(fig_p,f'F15_PCA_T{t}')
    fig_u.suptitle(f'UMAP | held-out Test {t} triangles; axes differ across panels');savefig(fig_u,f'F15_UMAP_T{t}')
    fig_l.suptitle(f'PCA coefficients | held-out Test {t} excluded from fitting');savefig(fig_l,f'F15_Loadings_T{t}')
    savefig(fig_e,f'F15_Error_Map_T{t}')
    print('Completed external Test',t)
pd.DataFrame(loads).to_csv(TABLE/'pca_coefficients.csv',index=False)
pd.DataFrame(variance).to_csv(TABLE/'pca_explained_variance.csv',index=False)
pd.DataFrame(drift).to_csv(TABLE/'feature_displacement.csv',index=False)
pd.DataFrame(coords).to_csv(TABLE/'projection_coordinates.csv.gz',index=False,compression='gzip')


In [ ]:
# 6. Readable results, environment, integrity manifest, ZIP and browser download.
notes=['# NB15 interpretation','',
'PCA/UMAP are exploratory, fitted only to development future-row records; external observations are transformed. All 18 channels are standardized using development data after the named representation.',
'PREFIX uses each domain initial 25 unique signatures; FULL_TARGET uses each domain complete unique-signature feature statistics and is transductive.',
'Classifiers were not refitted. Only RAW/PREFIX have NB14 predictions; SNV/FULL_TARGET are visual diagnostics here.',
'Vector-stage records can share a spectral state with conflicting labels; projections will coincide. No jitter is added to create artificial separation.',
'UMAP is an exploratory embedding; different fits have different coordinate systems. PCA coefficient sign is anchored only for display.',
'Row/signature resampling intervals are conditional sensitivity analyses on frozen predictions; signatures are not established independent biological units. Temporal and fruit-level dependence remain unresolved.',
'Leave-one-experiment-out summaries are descriptive across four outer folds with overlapping training sets. No population p-values are reported.',
'Use stage support and per-experiment predictions with all pooled summaries. Do not infer biochemical mechanisms from coefficients or causality from embeddings.','',metrics.to_string(index=False),'',loo.to_string(index=False)]
(RUN/'INTERPRETATION.md').write_text('\n\n'.join(notes),encoding='utf-8')
(META/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
record={'created_utc':datetime.now(timezone.utc).isoformat(),'input_nb14_zip':INPUT_ZIP.name,'input_nb14_sha256':sha(INPUT_ZIP),'master_sha256':sha(MASTER),'source_nb14_run':manifest['run_id'],'N':N,'bootstrap_draws':B,'seed':SEED,'classifier_refits':0,'python':platform.python_version(),'umap':umap.__version__,'n_evaluation_rows':len(wide),'analysis_role':'post-hoc descriptive diagnostics'}
(META/'run_manifest.json').write_text(json.dumps(record,indent=2))
h=[]
for path in sorted(RUN.rglob('*')):
    if path.is_file():h.append({'relative_path':str(path.relative_to(RUN)),'bytes':path.stat().st_size,'sha256':sha(path)})
pd.DataFrame(h).to_csv(META/'package_hashes.csv',index=False)
zip_path=Path(shutil.make_archive(str(RESULTS/('RESULTS_'+RUN.name)),'zip',root_dir=RUN.parent,base_dir=RUN.name))
with zipfile.ZipFile(zip_path) as zz:assert zz.testzip() is None
print('COMPLETADO. Adjunta este ZIP en el chat:',zip_path)
try:files.download(str(zip_path))
except Exception as e:print('Descarga automática no completada. El ZIP permanece en 06_RESULTS.',repr(e))
